# 02_analisis_SDO_GEE.ipynb
Análisis GEE: Tratamiento × SDO × Gap

**Sigue a:** `01_preprocesamiento.ipynb`  
**Input:** `dataset_between.csv`  
**Output:** tablas CSV con resultados de los modelos

### Columnas heredadas del preprocesamiento
| Variable | Descripción |
|---|---|
| `ID_Sujeto` | Identificador del participante |
| `Dilema` | Condición: `'Dist'`, `'Bloque_SIN'`, `'Bloque_CON'` |
| `Mantiene_bin` | VD: 1 = preserva jerarquía, 0 = revierte |
| `SDO_c` | SDO centrado en media muestral (ya procesado) |
| `NDC_c` | NDC centrado en media muestral (ya procesado) |
| `NDC_Score` | NDC sin centrar (para corte de subgrupo) |
| `Gap` | Brecha de desigualdad en ARS (0, 1000, 1200, 2000, 2400) |

---
## 0. Imports y configuración

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.genmod.generalized_estimating_equations import GEE
from statsmodels.genmod.families import Binomial
from statsmodels.genmod.cov_struct import Exchangeable
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.3f}'.format)
pd.set_option('display.max_columns', 20)

---
## 1. Carga del dataset analítico

In [2]:
df = pd.read_csv('dataset_between.csv')

print('Shape:', df.shape)
print('\nN sujetos:', df['ID_Sujeto'].nunique())
print('\nDistribución por condición (Dilema):')
print(df.groupby('Dilema')['ID_Sujeto'].nunique())
print('\nColumnas disponibles:')
print(df.dtypes)

Shape: (798, 42)

N sujetos: 133

Distribución por condición (Dilema):
Dilema
Bloque_CON    36
Bloque_SIN    47
Dist          50
Name: ID_Sujeto, dtype: int64

Columnas disponibles:
ID_Sujeto                   object
Origen_Form                 object
Identidad                   object
Dilema                      object
Orden_1                      int64
Bloque                      object
Orden_2                      int64
Respuesta                   object
Mantiene                     int64
SDO_Score                  float64
NDC_Score                  float64
expectativa_sin              int64
expectativa_grande           int64
expectativa_pequeña          int64
Genero                      object
politica                     int64
nivel_se                     int64
Gap_Size                   float64
expectativa_sin_num          int64
expectativa_grande_num       int64
expectativa_pequeña_num      int64
Expectativa_Activa         float64
Promedio_Gap_0.0           float64
Promedio_Gap_

---
## 2. Preparación de variables para el modelo

SDO_c y NDC_c ya vienen centrados del preprocesamiento.  
Solo se necesita crear Gap_k (Gap centrado en media muestral, en miles de ARS)
y las dummies de tratamiento.

In [3]:
# ── Verificar que SDO_c y NDC_c existen y están centrados ────────────────────
assert 'SDO_c' in df.columns, "Falta SDO_c — verificar que viene del preprocesamiento"
assert 'NDC_c' in df.columns, "Falta NDC_c — verificar que viene del preprocesamiento"
print(f'SDO_c:  M = {df["SDO_c"].mean():.6f}  (esperado ≈ 0.0)')
print(f'NDC_c:  M = {df["NDC_c"].mean():.6f}  (esperado ≈ 0.0)')
print()

# ── Gap centrado en media muestral, expresado en miles de ARS ────────────────
# Se centra sobre TODOS los ensayos (incluyendo Gap=0) igual que en modelos_sdo_gap
M_Gap = df['Gap'].mean()
df['Gap_k'] = (df['Gap'] - M_Gap) / 1000
print(f'Gap_k:  M = {df["Gap_k"].mean():.6f}  (esperado ≈ 0.0)')
print(f'Media Gap (ARS): {M_Gap:.1f}')
print()

# ── Dummies de tratamiento (referencia = Dist) ────────────────────────────────
df['T_CON'] = (df['Dilema'] == 'Bloque_CON').astype(int)
df['T_SIN'] = (df['Dilema'] == 'Bloque_SIN').astype(int)

print('Distribución dummies de tratamiento:')
print(df.groupby('Dilema')[['T_CON','T_SIN']].first())

# ── Ordenar por ID_Sujeto (requerimiento de GEE en statsmodels) ───────────────
df = df.sort_values('ID_Sujeto').reset_index(drop=True)

SDO_c:  M = -0.000000  (esperado ≈ 0.0)
NDC_c:  M = 0.000000  (esperado ≈ 0.0)

Gap_k:  M = -0.000000  (esperado ≈ 0.0)
Media Gap (ARS): 1100.0

Distribución dummies de tratamiento:
            T_CON  T_SIN
Dilema                  
Bloque_CON      1      0
Bloque_SIN      0      1
Dist            0      0


---
## 3. Estadística descriptiva de variables individuales (Tabla X)

In [4]:
df_suj = df.drop_duplicates(subset='ID_Sujeto').copy()

# Variables a reportar — ajustar si alguna no está en el dataset
vars_desc = {
    'NDC_Score'   : 'NDC',
    'SDO_Score'   : 'SDO',
    'politica'    : 'Ideología Izquierda-Derecha',
    'nivel_se'    : 'Nivel Socioeconómico Subjetivo'
}

rows = []
for col, label in vars_desc.items():
    if col in df_suj.columns:
        s = df_suj[col].dropna()
        rows.append({
            'Medida' : label,
            'Rango'  : f'{s.min():.2f}/{s.max():.2f}',
            'Media'  : round(s.mean(), 2),
            'DS'     : round(s.std(), 2)
        })

tabla_desc = pd.DataFrame(rows).set_index('Medida')
print('Tabla X — Estadística descriptiva')
print(tabla_desc)

# Dato adicional para reportar en texto: % con SDO < 2.0
sdo_vals = df_suj['SDO_Score']
print(f'\nSDO < 2.0: {(sdo_vals < 2.0).mean()*100:.1f}% de la muestra')
print(f'Asimetría SDO: {sdo_vals.skew():.3f}')

Tabla X — Estadística descriptiva
                                     Rango  Media    DS
Medida                                                 
NDC                              1.50/5.00  3.880 0.680
SDO                              1.00/4.20  2.120 0.750
Ideología Izquierda-Derecha      1.00/7.00  3.720 1.390
Nivel Socioeconómico Subjetivo  2.00/10.00  5.570 1.470

SDO < 2.0: 44.4% de la muestra
Asimetría SDO: 0.470


---
## 4. Existencia basal de ARR

### 4.1. Por condición — Tabla S4

In [5]:
def wilcoxon_vs_azar(serie, nombre=''):
    """
    Wilcoxon de rangos con signo contra 0.50.
    Entrada: serie de tasas por sujeto.
    """
    dif = serie - 0.50
    dif_nonzero = dif[dif != 0]
    if len(dif_nonzero) == 0:
        return dict(variable=nombre, n=len(serie),
                    M=serie.mean(), DS=serie.std(), W=np.nan, p=np.nan, sig='')
    W, p = stats.wilcoxon(dif_nonzero, alternative='two-sided')
    sig = ('***' if p < .001 else '**' if p < .01
           else '*' if p < .05 else '†' if p < .10 else '')
    return dict(variable=nombre, n=len(serie),
                M=round(serie.mean(), 3), DS=round(serie.std(), 3),
                W=round(W, 1), p=round(p, 3), sig=sig)

# Tasa de Mantiene por sujeto en su condición (primer bloque)
tasa_cond = (df.groupby(['ID_Sujeto', 'Dilema'])['Mantiene_bin']
               .mean()
               .reset_index()
               .rename(columns={'Mantiene_bin': 'tasa_mantiene'}))

# Etiquetas legibles para la tabla
etiqueta = {'Dist': 'DIST', 'Bloque_SIN': 'SIN', 'Bloque_CON': 'CON'}

rows = []
for cond_raw, cond_label in etiqueta.items():
    sub = tasa_cond[tasa_cond['Dilema'] == cond_raw]['tasa_mantiene']
    if len(sub) > 0:
        rows.append(wilcoxon_vs_azar(sub, nombre=cond_label))

tabla_s4 = pd.DataFrame(rows).set_index('variable')
print('Tabla S4 — Wilcoxon: Tasa Mantiene vs. azar (0.50) por condición')
print(tabla_s4)

Tabla S4 — Wilcoxon: Tasa Mantiene vs. azar (0.50) por condición
           n     M    DS       W     p  sig
variable                                   
DIST      50 0.347 0.291 172.500 0.001  ***
SIN       47 0.440 0.263 224.500 0.052    †
CON       36 0.324 0.276  63.500 0.000  ***


### 4.2. Por nivel de Gap — Tabla S5

In [6]:
tasa_gap = (df.groupby(['ID_Sujeto', 'Gap'])['Mantiene_bin']
              .mean()
              .reset_index()
              .rename(columns={'Mantiene_bin': 'tasa_mantiene'}))

rows = []
for gap_val in sorted(df['Gap'].unique()):
    sub = tasa_gap[tasa_gap['Gap'] == gap_val]['tasa_mantiene']
    rows.append(wilcoxon_vs_azar(sub, nombre=f'Gap={int(gap_val)}'))

tabla_s5 = pd.DataFrame(rows).set_index('variable')
print('Tabla S5 — Wilcoxon: Tasa Mantiene vs. azar (0.50) por nivel de Gap')
print(tabla_s5)

Tabla S5 — Wilcoxon: Tasa Mantiene vs. azar (0.50) por nivel de Gap
            n     M    DS        W     p  sig
variable                                     
Gap=0     133 0.571 0.395 1419.000 0.039    *
Gap=1000  133 0.376 0.486 3350.000 0.004   **
Gap=1200  133 0.323 0.470 2881.000 0.000  ***
Gap=2000  133 0.203 0.404 1809.000 0.000  ***
Gap=2400  133 0.195 0.398 1742.000 0.000  ***


---
## 5. Función auxiliar para formatear tablas GEE

In [7]:
def tabla_gee(resultado, titulo='Modelo GEE'):
    """
    Formatea resultados GEE en tabla con β, SE, OR, IC 95%, p.
    """
    params = resultado.params
    bse    = resultado.bse
    pvals  = resultado.pvalues
    ci     = resultado.conf_int()

    tbl = pd.DataFrame({
        'β'          : params.round(3),
        'SE'         : bse.round(3),
        'OR'         : np.exp(params).round(3),
        'IC_95_low'  : np.exp(ci[0]).round(3),
        'IC_95_high' : np.exp(ci[1]).round(3),
        'p'          : pvals.round(3),
    })
    tbl['sig'] = tbl['p'].apply(
        lambda p: '***' if p < .001 else
                  '**'  if p < .01  else
                  '*'   if p < .05  else
                  '†'   if p < .10  else ''
    )

    print(f'\n{titulo}')
    print('=' * 80)
    print(tbl.to_string())
    print(f'\nN observaciones : {int(resultado.nobs)}')
    print(f'N sujetos       : {resultado.model.num_group}')
    print(f'ρ exchangeable  : {resultado.cov_struct.dep_params:.4f}')
    return tbl

---
## 6. Modelo GEE — Muestra completa (Tabla 1)

**Especificación:** GEE binomial · link logit · correlación exchangeable ·  
estimador sandwich Huber-White · agrupado por ID_Sujeto.  
Interacción triple Tratamiento × SDO_c × Gap_k con todos los términos constitutivos.  
Referencia de tratamiento: condición Dist.

In [8]:
formula_principal = (
    'Mantiene_bin ~ T_CON + T_SIN + SDO_c + Gap_k '
    '+ T_CON:SDO_c + T_SIN:SDO_c '
    '+ T_CON:Gap_k + T_SIN:Gap_k '
    '+ SDO_c:Gap_k '
    '+ T_CON:SDO_c:Gap_k + T_SIN:SDO_c:Gap_k'
)

modelo_full = GEE.from_formula(
    formula    = formula_principal,
    groups     = 'ID_Sujeto',
    data       = df,
    family     = Binomial(),
    cov_struct = Exchangeable()
)

res_full = modelo_full.fit(cov_type='robust')

tbl_full = tabla_gee(
    res_full,
    titulo='Tabla 1 — GEE Tratamiento × SDO × Gap — Muestra completa (N=133, obs=798)'
)


Tabla 1 — GEE Tratamiento × SDO × Gap — Muestra completa (N=133, obs=798)
                       β    SE    OR  IC_95_low  IC_95_high     p  sig
Intercept         -0.683 0.201 0.505      0.341       0.749 0.001   **
T_CON             -0.167 0.317 0.846      0.455       1.574 0.599     
T_SIN              0.349 0.278 1.418      0.823       2.443 0.209     
SDO_c              0.051 0.313 1.052      0.570       1.943 0.871     
Gap_k             -0.656 0.163 0.519      0.377       0.714 0.000  ***
T_CON:SDO_c        0.074 0.461 1.077      0.437       2.656 0.872     
T_SIN:SDO_c        0.456 0.396 1.578      0.727       3.425 0.249     
T_CON:Gap_k       -0.157 0.262 0.855      0.512       1.428 0.550     
T_SIN:Gap_k       -0.270 0.257 0.763      0.461       1.264 0.294     
SDO_c:Gap_k       -0.157 0.194 0.855      0.585       1.249 0.417     
T_CON:SDO_c:Gap_k  0.111 0.307 1.117      0.612       2.041 0.718     
T_SIN:SDO_c:Gap_k  0.457 0.338 1.580      0.814       3.066 0.176     

N

---
## 7. Modelo GEE — Subgrupo NFC alta (Tabla 2)

**Criterio:** NDC_Score ≥ percentil 75 de la muestra analítica.  
Misma especificación que el modelo principal.  
**Análisis exploratorio** — post hoc, no confirmatorio.

In [9]:
# ── Definición del subgrupo ───────────────────────────────────────────────────
p75_ndc = df.drop_duplicates('ID_Sujeto')['NDC_Score'].quantile(0.75)
print(f'Percentil 75 NDC_Score: {p75_ndc:.4f}')

df_nfc = df[df['NDC_Score'] >= p75_ndc].copy()

n_suj_nfc = df_nfc['ID_Sujeto'].nunique()
n_obs_nfc = len(df_nfc)
print(f'Subgrupo: {n_suj_nfc} sujetos, {n_obs_nfc} observaciones')
print('\nDistribución por condición en subgrupo:')
print(df_nfc.groupby('Dilema')['ID_Sujeto'].nunique())
print('\n⚠ n_CON muy pequeño — coeficientes T_CON inestables por definición')

Percentil 75 NDC_Score: 4.3333
Subgrupo: 43 sujetos, 258 observaciones

Distribución por condición en subgrupo:
Dilema
Bloque_CON     7
Bloque_SIN    15
Dist          21
Name: ID_Sujeto, dtype: int64

⚠ n_CON muy pequeño — coeficientes T_CON inestables por definición


In [10]:
modelo_nfc = GEE.from_formula(
    formula    = formula_principal,
    groups     = 'ID_Sujeto',
    data       = df_nfc,
    family     = Binomial(),
    cov_struct = Exchangeable()
)

res_nfc = modelo_nfc.fit(cov_type='robust')

tbl_nfc = tabla_gee(
    res_nfc,
    titulo='Tabla 2 — GEE Tratamiento × SDO × Gap — Subgrupo NFC alta (exploratorio)'
)


Tabla 2 — GEE Tratamiento × SDO × Gap — Subgrupo NFC alta (exploratorio)
                       β    SE    OR  IC_95_low  IC_95_high     p sig
Intercept         -0.561 0.442 0.571      0.240       1.358 0.205    
T_CON             -0.667 0.758 0.513      0.116       2.270 0.379    
T_SIN             -0.115 0.561 0.891      0.297       2.673 0.837    
SDO_c             -0.374 0.596 0.688      0.214       2.211 0.530    
Gap_k             -0.476 0.259 0.621      0.374       1.033 0.066   †
T_CON:SDO_c        0.962 0.827 2.618      0.518      13.234 0.244    
T_SIN:SDO_c        1.449 0.771 4.258      0.940      19.282 0.060   †
T_CON:Gap_k       -0.116 0.450 0.891      0.369       2.150 0.797    
T_SIN:Gap_k       -0.219 0.403 0.804      0.364       1.772 0.588    
SDO_c:Gap_k        0.036 0.254 1.037      0.630       1.708 0.887    
T_CON:SDO_c:Gap_k  0.327 0.310 1.387      0.755       2.547 0.291    
T_SIN:SDO_c:Gap_k  0.517 0.530 1.677      0.594       4.737 0.329    

N observaciones

---
## 8. Diagnósticos

### 8.1. Distribución de SDO — Justificación del resultado nulo

In [11]:
sdo = df.drop_duplicates('ID_Sujeto')['SDO_Score']

print('Distribución SDO — muestra analítica (N=133 sujetos):')
print(sdo.describe(percentiles=[.10, .25, .50, .75, .90]).round(3))
print(f'\nAsimetría  : {sdo.skew():.3f}')
print(f'Curtosis   : {sdo.kurtosis():.3f}')
print(f'% SDO < 2.0: {(sdo < 2.0).mean()*100:.1f}%')
print(f'% SDO > 3.0: {(sdo > 3.0).mean()*100:.1f}%')
print()
print('Nota redacción:')
print(f'  M = {sdo.mean():.2f}, DS = {sdo.std():.2f}')
print(f'  44% con SDO < 2.0 → restricción severa de varianza para detectar moderaciones')

Distribución SDO — muestra analítica (N=133 sujetos):
count   133.000
mean      2.117
std       0.748
min       1.000
10%       1.200
25%       1.500
50%       2.100
75%       2.600
90%       3.180
max       4.200
Name: SDO_Score, dtype: float64

Asimetría  : 0.470
Curtosis   : -0.405
% SDO < 2.0: 44.4%
% SDO > 3.0: 11.3%

Nota redacción:
  M = 2.12, DS = 0.75
  44% con SDO < 2.0 → restricción severa de varianza para detectar moderaciones


### 8.2. Balance de celdas en subgrupo NFC

In [12]:
balance = (df_nfc.drop_duplicates('ID_Sujeto')
                 .groupby('Dilema')['ID_Sujeto']
                 .count()
                 .rename('n_sujetos'))
print('Balance por condición — subgrupo NFC alta:')
print(balance)
print()
n_con = balance.get('Bloque_CON', 0)
if n_con < 10:
    print(f'⚠ Bloque_CON: n={n_con} — cualquier inferencia sobre T_CON es inválida')

Balance por condición — subgrupo NFC alta:
Dilema
Bloque_CON     7
Bloque_SIN    15
Dist          21
Name: n_sujetos, dtype: int64

⚠ Bloque_CON: n=7 — cualquier inferencia sobre T_CON es inválida


---
## 9. Exportar resultados

In [13]:
tbl_full.to_csv('tabla1_GEE_muestra_completa.csv')
tbl_nfc.to_csv('tabla2_GEE_subgrupo_NFC.csv')
tabla_s4.to_csv('tabla_S4_wilcoxon_condicion.csv')
tabla_s5.to_csv('tabla_S5_wilcoxon_gap.csv')

print('Archivos exportados:')
for f in ['tabla1_GEE_muestra_completa.csv',
          'tabla2_GEE_subgrupo_NFC.csv',
          'tabla_S4_wilcoxon_condicion.csv',
          'tabla_S5_wilcoxon_gap.csv']:
    print(f'  {f}')

Archivos exportados:
  tabla1_GEE_muestra_completa.csv
  tabla2_GEE_subgrupo_NFC.csv
  tabla_S4_wilcoxon_condicion.csv
  tabla_S5_wilcoxon_gap.csv
